# Testing ondeviceMNIST inference live on webcam-feed


These bitfiles are from the MNIST_CNN AdaptiveHP, though one FixedHP is included.

In [59]:
#bitfile_path = 'bitfile_axi_master/system.bit'
#bitfile_path = 'bitfile_axi_stream/system.bit'
bitfile_path = 'bitfile_FixedHP_axi_master/system.bit'

shape_img = (1, 28, 28, 1) # 1 frame, 28 width, 28 height, 1 channel
shape_y = (1, 10) # 1 frame, digits from 0 to 9 

Load the model to the FPGA fabric (PL)

In [60]:
import numpy as np
# import the library/driver which is common for every VItis Unified synthesis
from bitfile_axi_master.axi_master_driver import NeuralNetworkOverlay
#from bitfile_axi_stream.axi_stream_driver import NeuralNetworkOverlay

# create the overlay object
overlay = NeuralNetworkOverlay(bitfile_name=bitfile_path, x_shape=shape_img, y_shape=shape_y, dtype=np.float32)

The jupyter notebook is run as root. To open GUIs we need to configure the environment. In user terminal, run the following to get the display:

```
env | grep DISPLAY
```
Camera is 30 FPS

In [76]:
import ipywidgets as widgets
from IPython.display import display
import cv2

threshold_slider = widgets.IntSlider(
    value=120, min=0, max=255, step=1, description="Threshold"
 )

interp_options = [
    ("INTER_NEAREST", cv2.INTER_NEAREST),
    ("INTER_LINEAR", cv2.INTER_LINEAR),
    ("INTER_CUBIC", cv2.INTER_CUBIC),
    ("INTER_AREA", cv2.INTER_AREA),
    ("INTER_LANCZOS4", cv2.INTER_LANCZOS4),
    ("INTER_LINEAR_EXACT", cv2.INTER_LINEAR_EXACT),
    ("INTER_NEAREST_EXACT", cv2.INTER_NEAREST_EXACT),
 ]
resize_interp_dropdown = widgets.Dropdown(
    options=interp_options,
    value=cv2.INTER_NEAREST,
    description="Resize interp"
 )

display(
    threshold_slider,
    resize_interp_dropdown,
 )

IntSlider(value=120, description='Threshold', max=255)

Dropdown(description='Resize interp', options=(('INTER_NEAREST', 0), ('INTER_LINEAR', 1), ('INTER_CUBIC', 2), …

In [ ]:
import os
import asyncio
import cv2
os.environ["DISPLAY"] = ":1"
#os.environ["XAUTHORITY"] = "/home/ubuntu/.Xauthority"
#os.environ["XAUTHORITY"] = "/run/user/1000/gdm/Xauthority"
#os.environ["DBUS_SESSION_BUS_ADDRESS"] = "unix:path=/run/user/1000/bus"

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Convert full frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Resize full frame to 28x28
    resized = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)#, interpolation=resize_interp_dropdown.value)

    # Threshold for cleaner digit (better than simple invert)
    _, resized = cv2.threshold(resized, int(threshold_slider.value), 255, cv2.THRESH_BINARY_INV)
    #_, resized = cv2.threshold(resized, 120#int(threshold_slider.value)
    #                           , 255, cv2.THRESH_TOZERO_INV) # https://docs.opencv.org/3.4/d7/d1b/group__imgproc__misc.html#ggaa9e58d2860d4afa658ef70a9b1115576a0e50a338a4b711a8c48f06a6b105dd98
    # Only invert
    #resized = cv2.bitwise_not(resized)

    # Normalize
    resized = resized.astype("float32") / 255.0

    # Reshape to (1,28,28,1)
    input_img = resized.reshape(1, 28, 28, 1)

    # Predict
    #prediction = model.predict(input_img, verbose=0)
    # Do the prediction/run inference
    result = overlay.predict(input_img, debug=False, profile=True, encode=np.float32, decode=np.float32)
    digit = np.argmax(result[0])
    confidence = np.max(result[0])

    # Display prediction
    cv2.putText(frame, 
                f"Predicted: {digit} ({confidence:.2f}) in {(result[1]) * 1e3:.3f} ms",
                (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,0,255), 2)

    cv2.imshow("Live feed", frame)
    cv2.imshow("MNIST processed feed", resized)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


QFontDatabase: Cannot find font directory /usr/local/share/pynq-venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /usr/local/share/pynq-venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /usr/local/share/pynq-venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /usr/local/share/pynq-venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /usr/local/sha

....

In [28]:
result

(PynqBuffer([[-4.96875 ,  0.65625 , -2.1875  , -1.875   ,  0.765625,
              -2.96875 , -4.53125 , -5.0625  , -1.4375  , -3.734375]],
            dtype=float32),
 0.0005695660001947545,
 1755.7227777958396)